# Test only Fidesz/1 with 32 workers

Run the cells below in your existing Roihu interactive session. The three processor files and six templates are unchanged. The launcher uses temporary links so the processor's input-deletion line cannot delete your original videos.

Only input subtree: `/scratch/project_2009497/Hungary_Data_Test/Hungary_Organised/Fidesz/1`

All dates and both platforms below this folder are included. Other participants are excluded.

Output: `/scratch/project_2009497/Hungary_Data_Test/Hungary_Splits_Fidesz1_Test/run_<timestamp>/Party/Participant/Date/Platform/VideoStem/`

See `README.md` for dependencies, outputs and verification limits.


In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys

RUNNER = Path("run_hungary_splits.py").resolve()
assert RUNNER.is_file(), "Open this notebook from the extracted hungary_fidesz1_test_32 folder."

def launch(*arguments):
    command = [sys.executable, "-u", str(RUNNER), "--workers", "32", "--scope", "Fidesz/1", *arguments]
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        # Stop only this launcher's newly created process group.
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        print("Splitting interrupted. Originals remain; this run may contain partial splits.")
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise RuntimeError(f"Launcher exited with code {returncode}. Read the messages/report above.")


## Check the setup

This checks imports, templates, the Fidesz/1 input scope and access to 32 CPUs. Confirm the printed `ONLY PROCESSING` path ends in `/Fidesz/1`. It does not start splitting.

In [ ]:
launch("--check-only")


## Run the splitting

The next cell starts 32 video-worker processes. Each invocation creates a fresh run directory. A status line appears every 30 seconds while waiting and a completion line appears after each recording. Detailed logs and `summary.csv` are saved inside the run directory.


In [ ]:
launch()
